In [1]:
# Cell 1：基础环境

from pathlib import Path
import json

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


SEED = 42

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


FINAL_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "distilbert_final"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

d:\anaconda\envs\pytorch_ecg\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
# Cell 2：加载最终DistilBERT和Tokenizer

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        FINAL_MODEL_DIR
    )
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        FINAL_MODEL_DIR
    )
)

model = model.to(device)

model.eval()

print(
    "Number of labels:",
    model.config.num_labels
)

print(
    "Model loaded from:",
    FINAL_MODEL_DIR
)

Number of labels: 77
Model loaded from: c:\Users\LYG Y9000x\OneDrive\Desktop\lyg master ai project\customer-ticket-routing\models\distilbert_final


In [3]:
# Cell 3：读取模型标签映射

id_to_label = {
    int(index): label
    for index, label
    in model.config.id2label.items()
}

label_to_id = {
    label: int(index)
    for label, index
    in model.config.label2id.items()
}


print(
    "Number of labels:",
    len(label_to_id)
)

print()

for index in range(10):
    print(
        index,
        "->",
        id_to_label[index]
    )

Number of labels: 77

0 -> Refund_not_showing_up
1 -> activate_my_card
2 -> age_limit
3 -> apple_pay_or_google_pay
4 -> atm_support
5 -> automatic_top_up
6 -> balance_not_updated_after_bank_transfer
7 -> balance_not_updated_after_cheque_or_cash_deposit
8 -> beneficiary_not_allowed
9 -> cancel_transfer


In [4]:
# Cell 4：77类Intent映射到业务部门

DEPARTMENT_MAPPING = {

    # ==================================
    # Card Services
    # ==================================

    "activate_my_card":
        "Card Services",

    "card_about_to_expire":
        "Card Services",

    "card_acceptance":
        "Card Services",

    "card_arrival":
        "Card Services",

    "card_delivery_estimate":
        "Card Services",

    "card_linking":
        "Card Services",

    "card_not_working":
        "Card Services",

    "card_swallowed":
        "Card Services",

    "change_pin":
        "Card Services",

    "compromised_card":
        "Card Services",

    "contactless_not_working":
        "Card Services",

    "get_physical_card":
        "Card Services",

    "getting_spare_card":
        "Card Services",

    "lost_or_stolen_card":
        "Card Services",

    "order_physical_card":
        "Card Services",

    "pin_blocked":
        "Card Services",

    "visa_or_mastercard":
        "Card Services",


    # ==================================
    # Virtual Cards
    # ==================================

    "disposable_card_limits":
        "Card Services",

    "get_disposable_virtual_card":
        "Card Services",

    "getting_virtual_card":
        "Card Services",

    "virtual_card_not_working":
        "Card Services",


    # ==================================
    # Card Payments / Refunds
    # ==================================

    "card_payment_fee_charged":
        "Card Payments",

    "card_payment_not_recognised":
        "Card Payments",

    "card_payment_wrong_exchange_rate":
        "Card Payments",

    "declined_card_payment":
        "Card Payments",

    "pending_card_payment":
        "Card Payments",

    "Refund_not_showing_up":
        "Card Payments",

    "request_refund":
        "Card Payments",

    "reverted_card_payment?":
        "Card Payments",

    "transaction_charged_twice":
        "Card Payments",

    "extra_charge_on_statement":
        "Card Payments",


    # ==================================
    # Transfers
    # ==================================

    "beneficiary_not_allowed":
        "Transfers",

    "cancel_transfer":
        "Transfers",

    "declined_transfer":
        "Transfers",

    "failed_transfer":
        "Transfers",

    "pending_transfer":
        "Transfers",

    "receiving_money":
        "Transfers",

    "transfer_fee_charged":
        "Transfers",

    "transfer_into_account":
        "Transfers",

    "transfer_not_received_by_recipient":
        "Transfers",

    "transfer_timing":
        "Transfers",

    "balance_not_updated_after_bank_transfer":
        "Transfers",


    # ==================================
    # Top Up & Deposits
    # ==================================

    "automatic_top_up":
        "Top Up & Deposits",

    "balance_not_updated_after_cheque_or_cash_deposit":
        "Top Up & Deposits",

    "pending_top_up":
        "Top Up & Deposits",

    "top_up_by_bank_transfer_charge":
        "Top Up & Deposits",

    "top_up_by_card_charge":
        "Top Up & Deposits",

    "top_up_by_cash_or_cheque":
        "Top Up & Deposits",

    "top_up_failed":
        "Top Up & Deposits",

    "top_up_limits":
        "Top Up & Deposits",

    "top_up_reverted":
        "Top Up & Deposits",

    "topping_up_by_card":
        "Top Up & Deposits",

    "verify_top_up":
        "Top Up & Deposits",

    "supported_cards_and_currencies":
        "Top Up & Deposits",


    # ==================================
    # Cash & ATM
    # ==================================

    "atm_support":
        "Cash & ATM",

    "cash_withdrawal_charge":
        "Cash & ATM",

    "cash_withdrawal_not_recognised":
        "Cash & ATM",

    "declined_cash_withdrawal":
        "Cash & ATM",

    "pending_cash_withdrawal":
        "Cash & ATM",

    "wrong_amount_of_cash_received":
        "Cash & ATM",

    "wrong_exchange_rate_for_cash_withdrawal":
        "Cash & ATM",


    # ==================================
    # Currency & Exchange
    # ==================================

    "exchange_charge":
        "Currency & Exchange",

    "exchange_rate":
        "Currency & Exchange",

    "exchange_via_app":
        "Currency & Exchange",

    "fiat_currency_support":
        "Currency & Exchange",

    "country_support":
        "Currency & Exchange",


    # ==================================
    # Account / Identity / Compliance
    # ==================================

    "age_limit":
        "Account & Identity",

    "edit_personal_details":
        "Account & Identity",

    "lost_or_stolen_phone":
        "Account & Identity",

    "passcode_forgotten":
        "Account & Identity",

    "terminate_account":
        "Account & Identity",

    "unable_to_verify_identity":
        "Account & Identity",

    "verify_my_identity":
        "Account & Identity",

    "verify_source_of_funds":
        "Account & Identity",

    "why_verify_identity":
        "Account & Identity",


    # ==================================
    # Direct Debit
    # ==================================

    "direct_debit_payment_not_recognised":
        "Card Payments",


    # ==================================
    # Digital Wallet
    # ==================================

    "apple_pay_or_google_pay":
        "Digital Wallet & General Banking",
}

In [5]:
# Cell 5：检查Department Mapping覆盖率

model_categories = set(
    label_to_id.keys()
)

mapped_categories = set(
    DEPARTMENT_MAPPING.keys()
)


missing_categories = (
    model_categories
    - mapped_categories
)

extra_categories = (
    mapped_categories
    - model_categories
)


print(
    "Model categories:",
    len(model_categories)
)

print(
    "Mapped categories:",
    len(mapped_categories)
)

print(
    "Missing:",
    missing_categories
)

print(
    "Extra:",
    extra_categories
)

Model categories: 77
Mapped categories: 77
Missing: set()
Extra: set()


In [6]:
assert len(
    missing_categories
) == 0

assert len(
    extra_categories
) == 0

print(
    "All 77 intents have been mapped successfully."
)

All 77 intents have been mapped successfully.


In [7]:
# Cell 6：读取冻结的数据划分

validation_df = pd.read_csv(
    PROCESSED_DATA_DIR
    / "validation_77.csv"
)

test_df = pd.read_csv(
    PROCESSED_DATA_DIR
    / "test_77.csv"
)

print(
    "Validation:",
    validation_df.shape
)

print(
    "Test:",
    test_df.shape
)

Validation: (1961, 2)
Test: (1961, 2)


In [8]:
# Cell 7：DistilBERT批量预测 + Confidence

def predict_with_confidence(
    texts,
    model,
    tokenizer,
    device,
    batch_size=32,
    max_length=64
):

    all_pred_ids = []
    all_confidences = []
    all_top2_confidences = []
    all_probability_margins = []

    model.eval()

    for start_index in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start_index:
            start_index + batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }

        with torch.no_grad():

            outputs = model(
                **encoded
            )

            probabilities = F.softmax(
                outputs.logits,
                dim=1
            )

        top2_probs, top2_ids = (
            torch.topk(
                probabilities,
                k=2,
                dim=1
            )
        )

        top1_probs = (
            top2_probs[:, 0]
        )

        second_probs = (
            top2_probs[:, 1]
        )

        pred_ids = (
            top2_ids[:, 0]
        )

        probability_margins = (
            top1_probs
            -
            second_probs
        )

        all_pred_ids.extend(
            pred_ids
            .cpu()
            .numpy()
        )

        all_confidences.extend(
            top1_probs
            .cpu()
            .numpy()
        )

        all_top2_confidences.extend(
            second_probs
            .cpu()
            .numpy()
        )

        all_probability_margins.extend(
            probability_margins
            .cpu()
            .numpy()
        )

    return {
        "pred_ids":
            np.array(all_pred_ids),

        "confidence":
            np.array(all_confidences),

        "second_confidence":
            np.array(all_top2_confidences),

        "probability_margin":
            np.array(all_probability_margins)
    }

In [9]:
# Cell 8：Validation预测

validation_outputs = (
    predict_with_confidence(
        texts=validation_df[
            "text"
        ].tolist(),

        model=model,
        tokenizer=tokenizer,
        device=device,
        batch_size=32,
        max_length=64
    )
)

In [10]:
validation_predictions = [
    id_to_label[
        int(pred_id)
    ]
    for pred_id
    in validation_outputs[
        "pred_ids"
    ]
]

In [11]:
validation_routing_df = (
    validation_df
    .reset_index(drop=True)
    .copy()
)

validation_routing_df[
    "predicted_category"
] = validation_predictions

validation_routing_df[
    "confidence"
] = validation_outputs[
    "confidence"
]

validation_routing_df[
    "second_confidence"
] = validation_outputs[
    "second_confidence"
]

validation_routing_df[
    "probability_margin"
] = validation_outputs[
    "probability_margin"
]

validation_routing_df[
    "is_correct"
] = (
    validation_routing_df[
        "category"
    ]
    ==
    validation_routing_df[
        "predicted_category"
    ]
)

In [12]:
# Cell 9：确认Validation结果

validation_accuracy = (
    validation_routing_df[
        "is_correct"
    ]
    .mean()
)

print(
    "Validation Accuracy:",
    f"{validation_accuracy:.4%}"
)

print(
    "Mean confidence:",
    f"{validation_routing_df['confidence'].mean():.4f}"
)

print(
    "Mean confidence - correct:",
    f"{validation_routing_df.loc[validation_routing_df['is_correct'], 'confidence'].mean():.4f}"
)

print(
    "Mean confidence - wrong:",
    f"{validation_routing_df.loc[~validation_routing_df['is_correct'], 'confidence'].mean():.4f}"
)

Validation Accuracy: 91.5859%
Mean confidence: 0.9520
Mean confidence - correct: 0.9715
Mean confidence - wrong: 0.7401


In [13]:
# Cell 10：Confidence描述统计

print(
    validation_routing_df[
        "confidence"
    ]
    .describe(
        percentiles=[
            0.10,
            0.20,
            0.30,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

count    1961.000000
mean        0.952008
std         0.118760
min         0.107656
10%         0.864589
20%         0.975559
30%         0.988374
50%         0.993441
75%         0.995547
90%         0.996435
95%         0.996814
max         0.997612
Name: confidence, dtype: float64


In [14]:
print(
    validation_routing_df.loc[
        ~validation_routing_df[
            "is_correct"
        ],
        "confidence"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

count    165.000000
mean       0.740103
std        0.218080
min        0.107656
10%        0.401689
25%        0.595230
50%        0.790849
75%        0.927291
90%        0.976321
max        0.995577
Name: confidence, dtype: float64


In [15]:
# Cell 11：评估一个Confidence Threshold

def evaluate_confidence_threshold(
    dataframe,
    threshold
):

    auto_mask = (
        dataframe["confidence"]
        >= threshold
    )

    manual_mask = (
        ~auto_mask
    )

    total_samples = len(
        dataframe
    )

    auto_samples = (
        auto_mask.sum()
    )

    manual_samples = (
        manual_mask.sum()
    )

    auto_coverage = (
        auto_samples
        / total_samples
    )

    manual_review_rate = (
        manual_samples
        / total_samples
    )

    if auto_samples > 0:

        auto_accuracy = (
            dataframe.loc[
                auto_mask,
                "is_correct"
            ]
            .mean()
        )

    else:

        auto_accuracy = np.nan

    total_errors = (
        ~dataframe[
            "is_correct"
        ]
    ).sum()

    captured_errors = (
        (
            ~dataframe[
                "is_correct"
            ]
        )
        &
        manual_mask
    ).sum()

    if total_errors > 0:

        error_capture_rate = (
            captured_errors
            / total_errors
        )

    else:

        error_capture_rate = np.nan

    return {
        "threshold":
            threshold,

        "auto_coverage":
            auto_coverage,

        "manual_review_rate":
            manual_review_rate,

        "auto_accuracy":
            auto_accuracy,

        "error_capture_rate":
            error_capture_rate,

        "auto_samples":
            auto_samples,

        "manual_samples":
            manual_samples
    }

In [16]:
# Cell 12：粗粒度Confidence阈值扫描

confidence_thresholds = [
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    0.97,
    0.99
]

threshold_results = []

for threshold in confidence_thresholds:

    result = (
        evaluate_confidence_threshold(
            validation_routing_df,
            threshold
        )
    )

    threshold_results.append(
        result
    )


threshold_results_df = (
    pd.DataFrame(
        threshold_results
    )
)


display(
    threshold_results_df
)

,threshold,auto_coverage,manual_review_rate,auto_accuracy,error_capture_rate,auto_samples,manual_samples
0,0.50,0.980622,0.019378,0.927197,0.151515,1923,38
1,0.60,0.960224,0.039776,0.935210,0.260606,1883,78
2,0.70,0.939827,0.060173,0.945198,0.387879,1843,118
3,0.75,0.933197,0.066803,0.950273,0.448485,1830,131
4,0.80,0.924018,0.075982,0.955298,0.509091,1812,149
5,0.85,0.903621,0.096379,0.959932,0.569697,1772,189
6,0.90,0.884243,0.115757,0.969435,0.678788,1734,227
7,0.95,0.849057,0.150943,0.980781,0.806061,1665,296
8,0.97,0.815400,0.184600,0.985616,0.860606,1599,362
9,0.99,0.665477,0.334523,0.993870,0.951515,1305,656


In [17]:
# Cell 13：精确寻找95%自动路由准确率阈值

TARGET_AUTO_ACCURACY = 0.95

candidate_thresholds = np.sort(
    validation_routing_df[
        "confidence"
    ]
    .unique()
)

fine_threshold_results = []

for threshold in candidate_thresholds:

    result = (
        evaluate_confidence_threshold(
            validation_routing_df,
            threshold
        )
    )

    fine_threshold_results.append(
        result
    )


fine_threshold_results_df = (
    pd.DataFrame(
        fine_threshold_results
    )
)


valid_thresholds_df = (
    fine_threshold_results_df[
        fine_threshold_results_df[
            "auto_accuracy"
        ]
        >= TARGET_AUTO_ACCURACY
    ]
    .copy()
)


best_threshold_row = (
    valid_thresholds_df
    .sort_values(
        by=[
            "auto_coverage",
            "threshold"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[0]
)


display(
    best_threshold_row
)

threshold                0.752886
auto_coverage            0.933197
manual_review_rate       0.066803
auto_accuracy            0.950273
error_capture_rate       0.448485
auto_samples          1830.000000
manual_samples         131.000000
Name: 131, dtype: float64

In [18]:
DISTILBERT_CONFIDENCE_THRESHOLD = float(
    best_threshold_row[
        "threshold"
    ]
)

print(
    "Selected threshold:",
    DISTILBERT_CONFIDENCE_THRESHOLD
)

print(
    "Auto coverage:",
    f"{best_threshold_row['auto_coverage']:.2%}"
)

print(
    "Manual review rate:",
    f"{best_threshold_row['manual_review_rate']:.2%}"
)

print(
    "Auto-routed accuracy:",
    f"{best_threshold_row['auto_accuracy']:.2%}"
)

print(
    "Error capture rate:",
    f"{best_threshold_row['error_capture_rate']:.2%}"
)

Selected threshold: 0.7528857588768005
Auto coverage: 93.32%
Manual review rate: 6.68%
Auto-routed accuracy: 95.03%
Error capture rate: 44.85%


In [19]:
# Cell 14：冻结最终Routing Threshold

DISTILBERT_CONFIDENCE_THRESHOLD = (
    0.7528857588768005
)

print(
    "Frozen confidence threshold:",
    DISTILBERT_CONFIDENCE_THRESHOLD
)

Frozen confidence threshold: 0.7528857588768005


In [20]:
# Cell 15：Test set最终Routing评估

test_outputs = (
    predict_with_confidence(
        texts=test_df[
            "text"
        ].tolist(),

        model=model,
        tokenizer=tokenizer,
        device=device,
        batch_size=32,
        max_length=64
    )
)


test_predictions = [
    id_to_label[
        int(pred_id)
    ]
    for pred_id
    in test_outputs[
        "pred_ids"
    ]
]


test_routing_df = (
    test_df
    .reset_index(drop=True)
    .copy()
)


test_routing_df[
    "predicted_category"
] = test_predictions

test_routing_df[
    "confidence"
] = test_outputs[
    "confidence"
]

test_routing_df[
    "is_correct"
] = (
    test_routing_df[
        "category"
    ]
    ==
    test_routing_df[
        "predicted_category"
    ]
)


test_routing_result = (
    evaluate_confidence_threshold(
        test_routing_df,
        DISTILBERT_CONFIDENCE_THRESHOLD
    )
)


print(
    "Raw Test Accuracy:",
    f"{test_routing_df['is_correct'].mean():.2%}"
)

print(
    "Auto Coverage:",
    f"{test_routing_result['auto_coverage']:.2%}"
)

print(
    "Manual Review Rate:",
    f"{test_routing_result['manual_review_rate']:.2%}"
)

print(
    "Auto-routed Accuracy:",
    f"{test_routing_result['auto_accuracy']:.2%}"
)

print(
    "Error Capture Rate:",
    f"{test_routing_result['error_capture_rate']:.2%}"
)

Raw Test Accuracy: 92.40%
Auto Coverage: 94.34%
Manual Review Rate: 5.66%
Auto-routed Accuracy: 95.41%
Error Capture Rate: 42.95%


In [21]:
# Cell 16：最终单条工单预测接口

def predict_ticket(
    text,
    model,
    tokenizer,
    device,
    department_mapping,
    confidence_threshold,
    max_length=64
):

    if not isinstance(
        text,
        str
    ):

        raise TypeError(
            "Ticket text must be a string."
        )

    if not text.strip():

        raise ValueError(
            "Ticket text cannot be empty."
        )


    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    encoded = {
        key: value.to(device)
        for key, value
        in encoded.items()
    }


    model.eval()

    with torch.no_grad():

        outputs = model(
            **encoded
        )

        probabilities = (
            F.softmax(
                outputs.logits,
                dim=1
            )
        )


    top2_probs, top2_ids = (
        torch.topk(
            probabilities,
            k=2,
            dim=1
        )
    )


    top1_id = int(
        top2_ids[0, 0].item()
    )

    top2_id = int(
        top2_ids[0, 1].item()
    )


    confidence = float(
        top2_probs[
            0, 0
        ].item()
    )

    second_confidence = float(
        top2_probs[
            0, 1
        ].item()
    )


    predicted_intent = (
        id_to_label[
            top1_id
        ]
    )

    alternative_intent = (
        id_to_label[
            top2_id
        ]
    )


    department = (
        department_mapping[
            predicted_intent
        ]
    )


    manual_review = (
        confidence
        <
        confidence_threshold
    )


    routing_decision = (
        "MANUAL_REVIEW"
        if manual_review
        else "AUTO_ROUTE"
    )


    return {

        "text":
            text,

        "predicted_intent":
            predicted_intent,

        "department":
            department,

        "confidence":
            round(
                confidence,
                4
            ),

        "alternative_intent":
            alternative_intent,

        "alternative_confidence":
            round(
                second_confidence,
                4
            ),

        "confidence_margin":
            round(
                confidence
                - second_confidence,
                4
            ),

        "routing_decision":
            routing_decision,

        "manual_review":
            manual_review
    }

In [22]:
result = predict_ticket(

    text=(
        "My transfer has not arrived yet"
    ),

    model=model,

    tokenizer=tokenizer,

    device=device,

    department_mapping=(
        DEPARTMENT_MAPPING
    ),

    confidence_threshold=(
        DISTILBERT_CONFIDENCE_THRESHOLD
    )
)

result

{'text': 'My transfer has not arrived yet',
 'predicted_intent': 'balance_not_updated_after_bank_transfer',
 'department': 'Transfers',
 'confidence': 0.7487,
 'alternative_intent': 'transfer_not_received_by_recipient',
 'alternative_confidence': 0.2175,
 'confidence_margin': 0.5312,
 'routing_decision': 'MANUAL_REVIEW',
 'manual_review': True}

In [23]:
test_tickets = [

    "My card has stopped working.",

    "Why is my transfer still pending?",

    "I don't recognise this cash withdrawal.",

    "I need to change my phone number.",

    "Why do I need to verify my identity?"
]


for ticket in test_tickets:

    result = predict_ticket(
        text=ticket,
        model=model,
        tokenizer=tokenizer,
        device=device,
        department_mapping=(
            DEPARTMENT_MAPPING
        ),
        confidence_threshold=(
            DISTILBERT_CONFIDENCE_THRESHOLD
        )
    )

    print("=" * 80)

    print(
        "Ticket:",
        ticket
    )

    print(
        "Intent:",
        result[
            "predicted_intent"
        ]
    )

    print(
        "Department:",
        result[
            "department"
        ]
    )

    print(
        "Confidence:",
        result[
            "confidence"
        ]
    )

    print(
        "Decision:",
        result[
            "routing_decision"
        ]
    )

Ticket: My card has stopped working.
Intent: card_not_working
Department: Card Services
Confidence: 0.9872
Decision: AUTO_ROUTE
Ticket: Why is my transfer still pending?
Intent: pending_transfer
Department: Transfers
Confidence: 0.9935
Decision: AUTO_ROUTE
Ticket: I don't recognise this cash withdrawal.
Intent: cash_withdrawal_not_recognised
Department: Cash & ATM
Confidence: 0.9955
Decision: AUTO_ROUTE
Ticket: I need to change my phone number.
Intent: edit_personal_details
Department: Account & Identity
Confidence: 0.9748
Decision: AUTO_ROUTE
Ticket: Why do I need to verify my identity?
Intent: why_verify_identity
Department: Account & Identity
Confidence: 0.9886
Decision: AUTO_ROUTE


In [24]:
# Cell 18：保存最终系统配置

ROUTING_CONFIG = {

    "model":
        "distilbert-base-uncased",

    "num_classes":
        77,

    "max_length":
        64,

    "confidence_threshold":
        DISTILBERT_CONFIDENCE_THRESHOLD,

    "target_auto_accuracy":
        0.95,

    "validation_auto_coverage":
        float(
            best_threshold_row[
                "auto_coverage"
            ]
        ),

    "validation_manual_review_rate":
        float(
            best_threshold_row[
                "manual_review_rate"
            ]
        ),

    "validation_auto_accuracy":
        float(
            best_threshold_row[
                "auto_accuracy"
            ]
        ),

    "validation_error_capture_rate":
        float(
            best_threshold_row[
                "error_capture_rate"
            ]
        ),

    "department_mapping":
        DEPARTMENT_MAPPING
}


CONFIG_DIR = (
    PROJECT_ROOT
    / "config"
)

CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


with open(
    CONFIG_DIR
    / "routing_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        ROUTING_CONFIG,
        file,
        indent=4,
        ensure_ascii=False
    )


print(
    "Routing configuration saved."
)

Routing configuration saved.


In [25]:
# Cell 19：保存最终Test Routing结果

test_routing_df[
    "department"
] = test_routing_df[
    "predicted_category"
].map(
    DEPARTMENT_MAPPING
)

test_routing_df[
    "manual_review"
] = (
    test_routing_df["confidence"]
    <
    DISTILBERT_CONFIDENCE_THRESHOLD
)

test_routing_df[
    "routing_decision"
] = np.where(
    test_routing_df["manual_review"],
    "MANUAL_REVIEW",
    "AUTO_ROUTE"
)

test_routing_df.to_csv(
    RESULTS_DIR
    / "final_routing_test_results.csv",
    index=False
)

print(
    "Final routing results saved."
)

Final routing results saved.


In [26]:
# Cell 20：保存最终系统指标

final_system_metrics = {
    "raw_test_accuracy":
        float(
            test_routing_df[
                "is_correct"
            ].mean()
        ),

    "confidence_threshold":
        DISTILBERT_CONFIDENCE_THRESHOLD,

    "auto_coverage":
        float(
            test_routing_result[
                "auto_coverage"
            ]
        ),

    "manual_review_rate":
        float(
            test_routing_result[
                "manual_review_rate"
            ]
        ),

    "auto_routed_accuracy":
        float(
            test_routing_result[
                "auto_accuracy"
            ]
        ),

    "error_capture_rate":
        float(
            test_routing_result[
                "error_capture_rate"
            ]
        )
}

with open(
    RESULTS_DIR
    / "final_system_metrics.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_system_metrics,
        file,
        indent=4
    )

print(final_system_metrics)

{'raw_test_accuracy': 0.9240183579806222, 'confidence_threshold': 0.7528857588768005, 'auto_coverage': 0.9433962264150944, 'manual_review_rate': 0.05660377358490566, 'auto_routed_accuracy': 0.9540540540540541, 'error_capture_rate': 0.42953020134228187}
